In [ ]:
import json
import numpy as np
import pandas as pd
from typing import List, Tuple

# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------
CSV_PATH = "data-wvs.csv"
QMAP_PATH = "map-wvs.json"
COUNTRY_COL = "B_COUNTRY_ALPHA"
TARGET_COUNTRY = "Argentina"

# Optional: how many items to display for strengths/weaknesses in console
N_SHOW = 12

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------
def percentile_rank(values: np.ndarray, x: float) -> float:
    """Return the percentile rank of x in values (percentage <= x)."""
    vals = values[~np.isnan(values)]
    if vals.size == 0 or np.isnan(x):
        return np.nan
    return 100.0 * (np.sum(vals <= x) / vals.size)

def top_bottom(
    percentiles: dict, series: pd.Series, qmap: dict, n: int = 12
) -> Tuple[List[Tuple[str, float, float, str]], List[Tuple[str, float, float, str]]]:
    """Return top-n and bottom-n questions by percentile rank for the target country."""
    ranked = [(q, pr) for q, pr in percentiles.items() if not np.isnan(pr)]
    ranked.sort(key=lambda t: t[1], reverse=True)
    top = [(q, pr, float(series[q]), qmap.get(q, q)) for q, pr in ranked[:n]]
    ranked.sort(key=lambda t: t[1])
    bottom = [(q, pr, float(series[q]), qmap.get(q, q)) for q, pr in ranked[:n]]
    return top, bottom

def fmt_item(qcode: str, pr: float, value: float, label: str) -> str:
    return f"{qcode} ({pr:0.1f}th pctile, value={value:g}) – {label}"

# -----------------------------------------------------------------------------
# Load data
# -----------------------------------------------------------------------------
df = pd.read_csv(CSV_PATH)
with open(QMAP_PATH, "r", encoding="utf-8") as f:
    qmap = json.load(f)

# Identify question columns and ensure numeric
q_cols = [c for c in df.columns if c.startswith("Q")]
for c in q_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Check target availability
if TARGET_COUNTRY not in set(df[COUNTRY_COL]):
    raise ValueError(f"{TARGET_COUNTRY} not found in {COUNTRY_COL}")

# If multiple rows for country, take the mean across them
arg_df = df[df[COUNTRY_COL] == TARGET_COUNTRY]
arg_series = arg_df[q_cols].mean(numeric_only=True)

# -----------------------------------------------------------------------------
# Compute percentile ranks for Argentina vs. all countries
# -----------------------------------------------------------------------------
percentiles = {}
for q in q_cols:
    series = df[q].to_numpy(dtype=float)
    arg_val = arg_series.get(q, np.nan)
    if np.isnan(arg_val) or np.sum(~np.isnan(series)) < 5:
        continue
    percentiles[q] = percentile_rank(series, arg_val)

# -----------------------------------------------------------------------------
# Inspect top/bottom indicators
# -----------------------------------------------------------------------------
topN, bottomN = top_bottom(percentiles, arg_series, qmap, n=N_SHOW)

print("\n=== Argentina: Top indicators (highest percentile ranks) ===")
for q, pr, val, label in topN:
    print(" • " + fmt_item(q, pr, val, label))

print("\n=== Argentina: Bottom indicators (lowest percentile ranks) ===")
for q, pr, val, label in bottomN:
    print(" • " + fmt_item(q, pr, val, label))

# -----------------------------------------------------------------------------
# Build the 3-sentence EDA summary (strengths, weaknesses/average, conclusion)
# - You can tailor which questions feed each sentence using the selected_keys below.
# -----------------------------------------------------------------------------
# Pick a curated set of questions that are intuitive to interpret
selected_keys = {
    # Wellbeing / satisfaction
    "life_satisfaction": "Q49",   # 1–10 higher = better
    "financial_satisfaction": "Q50",  # 1–10 higher = better
    "happiness": "Q46",  # lower numbers ~ 'very/rather happy' ordering; use percentile carefully

    # Trust in institutions
    "police_conf": "Q69",
    "courts_conf": "Q70",
    "gov_conf": "Q71",
    "elections_conf": "Q76",

    # Democracy values
    "free_elections_essential": "Q243",
    "democracy_important": "Q250",
    "how_democratic_today": "Q251",

    # Corruption perceptions / security
    "corruption_level": "Q112",  # 1 no corruption – 10 abundant corruption (higher = worse)
    "feel_secure": "Q131",       # higher = more secure

    # Participation
    "vote_local": "Q221",
    "vote_national": "Q222",

    # Social distance items (lower value = fewer would exclude as neighbors)
    "deny_diff_religion_neighbor": "Q23",
    "deny_diff_race_neighbor": "Q19",
    "deny_immigrant_neighbor": "Q21",
}

def get(qcode: str, default=np.nan) -> float:
    return float(arg_series.get(qcode, default))

def pr(qcode: str, default=np.nan) -> float:
    return float(percentiles.get(qcode, default))

# Pull values & percentiles
vals = {k: get(q) for k, q in selected_keys.items()}
prs  = {k: pr(q)  for k, q in selected_keys.items()}

# Sentence 1: strengths (pick a few high-percentile, broadly positive indicators)
strength_bits = []
if not np.isnan(vals["life_satisfaction"]):
    strength_bits.append(f"high life satisfaction ({vals['life_satisfaction']:.2f}/10; ~{prs['life_satisfaction']:.0f}th percentile)")
if not np.isnan(prs["police_conf"]):
    strength_bits.append(f"strong confidence in the police (~{prs['police_conf']:.0f}th pctile)")
if not np.isnan(prs["courts_conf"]):
    strength_bits.append(f"and courts (~{prs['courts_conf']:.0f}th pctile)")
if not np.isnan(prs["free_elections_essential"]):
    strength_bits.append(f"with free elections seen as essential (~{prs['free_elections_essential']:.0f}th pctile)")

sentence1 = "Argentina’s standout strengths are " + ", ".join(strength_bits).replace(", and", " and") + "."

# Sentence 2: weaknesses / average areas (choose bottom or middling)
weak_bits = []
if not np.isnan(vals["corruption_level"]):
    weak_bits.append(f"perceived corruption is high ({vals['corruption_level']:.2f}/10; ~{prs['corruption_level']:.0f}th pctile)")
if not np.isnan(vals["financial_satisfaction"]):
    weak_bits.append(f"financial satisfaction is middling ({vals['financial_satisfaction']:.2f}/10; ~{prs['financial_satisfaction']:.0f}th pctile)")
if not np.isnan(prs["vote_national"]):
    weak_bits.append(f"self‑reported voting rates are low (~{prs['vote_national']:.0f}th pctile)")
# Social distance measures – high percentile here means more exclusion (a weakness)
social_flags = []
for k in ["deny_diff_religion_neighbor", "deny_diff_race_neighbor", "deny_immigrant_neighbor"]:
    if not np.isnan(prs[k]) and prs[k] >= 90:
        social_flags.append(k)
if social_flags:
    weak_bits.append("exclusionary attitudes toward some out‑groups as neighbors are comparatively common")

sentence2 = "At the same time, " + "; ".join(weak_bits) + "."

# Sentence 3: concluding statement
sentence3 = "Overall, Argentina appears civically principled and institution‑confident, yet still contends with corruption, social inclusion, and pocketbook pressures."

# Print the requested 3-sentence summary
print("\n=== Three‑sentence EDA summary (Argentina) ===")
print(sentence1)
print(sentence2)
print(sentence3)
``